In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from tqdm import tqdm
import pandas as pd
import time

def scrape_match_details(url):
    try:
        chrome_options = Options()
        chrome_options.add_argument("--headless")
        chrome_options.add_argument('--no-sandbox')
        chrome_options.add_argument('--disable-dev-shm-usage')
        webdriver_service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=webdriver_service, options=chrome_options)
        driver.get(url)        
        wait = WebDriverWait(driver, 10)
        try:
            match_date = driver.find_element(By.CSS_SELECTOR, '.css-1tttqnj-MatchDateCSS').text
        except Exception as e:
            match_date = "Date not found"
            print(f"Error extracting date: {e}")
        try:
            team_elements = driver.find_elements(By.CSS_SELECTOR, '.css-dpbuul-TeamNameItself-TeamNameOnTabletUp')
            home_team = team_elements[0].text if len(team_elements) > 0 else "Home Team not found"
            away_team = team_elements[1].text if len(team_elements) > 1 else "Away Team not found"
        except Exception as e:
            home_team = "Home Team not found"
            away_team = "Away Team not found"
            print(f"Error extracting team names: {e}")
        try:
            added_time_elements = driver.find_elements(By.CSS_SELECTOR, '.css-1agmro5-AddedTime')
            added_minutes_45 = added_time_elements[0].text.strip('+').strip('minutes added') if added_time_elements else None
            added_minutes_90 = added_time_elements[1].text.strip('+').strip('minutes added') if len(added_time_elements) > 1 else None
        except Exception as e:
            added_minutes_45 = None
            added_minutes_90 = None
            print(f"Error extracting added time: {e}")
        driver.quit()
        return {
            'URL': url,
            'Date': match_date,
            'Home Team': home_team,
            'Away Team': away_team,
            '45+': added_minutes_45,
            '90+': added_minutes_90
        }
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return {}
base_url = 'https://www.fotmob.com/matches/arsenal-vs-brentford/389pak#'
results = []
for link_id in tqdm(range(3609929, 3610310), desc="Scraping Match Details", unit="link"):
    try:
        url = f'{base_url}{link_id}'
        match_details = scrape_match_details(url)
        if match_details:
            results.append(match_details)
        time.sleep(0.1)    
    except Exception as e:
        print(f"Skipped link {link_id} due to error: {e}")
        continue

df = pd.DataFrame(results)
print(df.head())
df.to_csv('2022_23_comprehensive_match_details.csv', index=False)